In [ ]:
# AGI Bench: Cognitive Reflection Test
!pip install -q protobuf==5.29.6 kaggle-benchmarks numpy 2>/dev/null


## Cognitive Science Rationale

The **Cognitive Reflection Test** (Frederick, 2005) measures the tendency to override an intuitive but incorrect response with a deliberative correct one. It indexes the System 1 → System 2 transition (Kahneman, 2011).

Our version uses **procedurally generated variants** — not the famous 3 items — to prevent memorization. Human accuracy: 30–48%.


## Interpreting the Score

| Score | Interpretation |
|:---:|---|
| 0.8–1.0 | Strong System 2 override — resists intuitive traps |
| 0.5–0.8 | Mixed — sometimes falls for traps |
| 0.3–0.5 | Human-level performance |
| 0.0–0.3 | Dominated by System 1 heuristics |


### References
Frederick (2005), Kahneman (2011)


# 🧠 AGI Bench: Cognitive Reflection Test (CRT)

**Track:** Executive Functions  
**Construct:** Response Inhibition (System 1 vs System 2)  
**Theory:** Frederick (2005), Kahneman (2011), Miyake et al. (2000)  

Tests the ability to override compelling intuitive-but-wrong answers with deliberate reasoning. 
20 novel CRT-style questions with specific intuitive traps.

**Human baseline:** ~30% accuracy (general public), ~50% (MIT students)  
**Score:** 0.40 × accuracy + 0.30 × (1 - trap_rate) + 0.20 × difficulty_bonus + 0.10 × calibration


In [ ]:
"""
Cognitive Reflection Test (CRT) data items.

Novel items inspired by Frederick (2005) but using original questions
to avoid contamination from published CRT tests.

Each item has an intuitive-but-wrong answer (System 1 response) and
a correct answer requiring deliberate reasoning (System 2 override).

Design principle: The intuitive answer is COMPELLINGLY wrong — it feels
obviously right, requiring active inhibition to override.
"""

CRT_ITEMS = [
    {
        "id": "CRT01",
        "question": "A printer and a cable together cost $37. The printer costs $30 more than the cable. How much does the cable cost?",
        "intuitive_wrong": "7",
        "correct": "3.50",
        "answer_unit": "dollars",
        "explanation": "If cable = x, printer = x + 30. x + (x + 30) = 37 → 2x = 7 → x = 3.50",
        "difficulty": "easy",
        "cognitive_trap": "anchoring on the $30 difference",
    },
    {
        "id": "CRT02",
        "question": "If 3 robots can assemble 3 widgets in 3 minutes, how many minutes would it take 100 robots to assemble 100 widgets?",
        "intuitive_wrong": "100",
        "correct": "3",
        "answer_unit": "minutes",
        "explanation": "Each robot makes 1 widget in 3 minutes. 100 robots make 100 widgets in 3 minutes.",
        "difficulty": "easy",
        "cognitive_trap": "linear proportional scaling",
    },
    {
        "id": "CRT03",
        "question": "In a pond, there is a patch of algae. Every day, the patch doubles in size. If it takes 30 days for the patch to cover the entire pond, how many days does it take to cover half the pond?",
        "intuitive_wrong": "15",
        "correct": "29",
        "answer_unit": "days",
        "explanation": "Doubles daily; full on day 30 means half on day 29.",
        "difficulty": "easy",
        "cognitive_trap": "halving the time for half the result",
    },
    {
        "id": "CRT04",
        "question": "A farmer has 15 sheep. All but 8 run away. How many sheep does the farmer have left?",
        "intuitive_wrong": "7",
        "correct": "8",
        "answer_unit": "sheep",
        "explanation": "'All but 8' = 8 remain.",
        "difficulty": "medium",
        "cognitive_trap": "subtraction reflex (15 - 8)",
    },
    {
        "id": "CRT05",
        "question": "You're running a race and you overtake the person in 2nd place. What position are you now in?",
        "intuitive_wrong": "1",
        "correct": "2",
        "answer_unit": "position",
        "explanation": "Overtaking 2nd place puts you in 2nd, not 1st.",
        "difficulty": "medium",
        "cognitive_trap": "equating 'overtake' with 'move ahead of everyone'",
    },
    {
        "id": "CRT06",
        "question": "A store marks up all items by 25%, then offers a 25% discount. What is the net percentage change from the original price?",
        "intuitive_wrong": "0",
        "correct": "-6.25",
        "answer_unit": "percent",
        "explanation": "1.25 × 0.75 = 0.9375 → 6.25% decrease.",
        "difficulty": "hard",
        "cognitive_trap": "+25% and -25% seem to cancel out",
    },
    {
        "id": "CRT07",
        "question": "A clock takes 5 seconds to strike 6 o'clock (6 strikes). How many seconds does it take to strike 12 o'clock?",
        "intuitive_wrong": "10",
        "correct": "11",
        "answer_unit": "seconds",
        "explanation": "6 strikes = 5 gaps of 1s each. 12 strikes = 11 gaps = 11 seconds.",
        "difficulty": "hard",
        "cognitive_trap": "doubling strikes = doubling time",
    },
    {
        "id": "CRT08",
        "question": "A snail climbs 3 meters up a wall during the day but slides back 2 meters at night. The wall is 10 meters high. How many days does it take to reach the top?",
        "intuitive_wrong": "10",
        "correct": "8",
        "answer_unit": "days",
        "explanation": "After 7 days: 7m net. Day 8: climbs to 10m — done. No night slide needed.",
        "difficulty": "hard",
        "cognitive_trap": "10m ÷ 1m/day = 10 days",
    },
    {
        "id": "CRT09",
        "question": "Emily's mother has 4 children. The first is named April, the second is May, the third is June. What is the fourth child's name?",
        "intuitive_wrong": "July",
        "correct": "Emily",
        "answer_unit": "name",
        "explanation": "The question says 'Emily's mother' — the fourth child is Emily.",
        "difficulty": "medium",
        "cognitive_trap": "pattern continuation (month names)",
    },
    {
        "id": "CRT10",
        "question": "A bat and a ball cost $1.10 in total. The bat costs exactly $1.00 more than the ball. How much does the ball cost, in cents?",
        "intuitive_wrong": "10",
        "correct": "5",
        "answer_unit": "cents",
        "explanation": "ball = x, bat = x + 100 cents. x + x + 100 = 110 → x = 5 cents.",
        "difficulty": "easy",
        "cognitive_trap": "anchoring on $1.00 and $0.10",
    },
    {
        "id": "CRT11",
        "question": "You have a book with 100 pages. You tear out pages 7, 8, 23, 24, 95, and 96. How many individual LEAVES (physical sheets) did you remove?",
        "intuitive_wrong": "6",
        "correct": "3",
        "answer_unit": "leaves",
        "explanation": "Pages 7-8, 23-24, 95-96 are front/back of the same leaves. 3 leaves removed.",
        "difficulty": "hard",
        "cognitive_trap": "conflating pages with leaves",
    },
    {
        "id": "CRT12",
        "question": "How many times can you subtract 5 from 25?",
        "intuitive_wrong": "5",
        "correct": "1",
        "answer_unit": "times",
        "explanation": "After the first subtraction, you're subtracting from 20, not 25.",
        "difficulty": "medium",
        "cognitive_trap": "25 ÷ 5 = 5 (division instead of literal reading)",
    },
    {
        "id": "CRT13",
        "question": "A rope is cut into 3 pieces. How many cuts were made?",
        "intuitive_wrong": "3",
        "correct": "2",
        "answer_unit": "cuts",
        "explanation": "n pieces require n-1 cuts.",
        "difficulty": "easy",
        "cognitive_trap": "pieces = cuts",
    },
    {
        "id": "CRT14",
        "question": "A doctor gives you 3 pills and tells you to take one every half hour. How many minutes does it take to finish all the pills?",
        "intuitive_wrong": "90",
        "correct": "60",
        "answer_unit": "minutes",
        "explanation": "Take pill 1 at 0 min, pill 2 at 30 min, pill 3 at 60 min = 60 minutes total.",
        "difficulty": "medium",
        "cognitive_trap": "3 pills × 30 min = 90 min",
    },
    {
        "id": "CRT15",
        "question": "If there are 12 one-cent stamps in a dozen, how many two-cent stamps are in a dozen?",
        "intuitive_wrong": "6",
        "correct": "12",
        "answer_unit": "stamps",
        "explanation": "A dozen is always 12, regardless of stamp denomination.",
        "difficulty": "medium",
        "cognitive_trap": "dividing by the denomination",
    },
    {
        "id": "CRT16",
        "question": "Two trains start 200 km apart and travel toward each other, each at 50 km/h. A fly starts at one train and flies back and forth between them at 75 km/h until they meet. How far does the fly travel?",
        "intuitive_wrong": "complicated",
        "correct": "150",
        "answer_unit": "km",
        "explanation": "Trains meet in 200/(50+50) = 2 hours. Fly travels 75 × 2 = 150 km.",
        "difficulty": "hard",
        "cognitive_trap": "trying to sum infinite bounces instead of using total time",
    },
    {
        "id": "CRT17",
        "question": "You have two hourglasses: one that measures 4 minutes and one that measures 7 minutes. You start both at the same time. When the 4-minute hourglass runs out, how many minutes are LEFT in the 7-minute hourglass?",
        "intuitive_wrong": "4",
        "correct": "3",
        "answer_unit": "minutes",
        "explanation": "7 - 4 = 3 minutes remaining. The intuitive trap is answering '4' (the hourglass that just ran out).",
        "difficulty": "easy",
        "cognitive_trap": "echoing the recently mentioned number (4) instead of computing the difference",
    },
    {
        "id": "CRT18",
        "question": "A brick weighs 1 kg plus half a brick. How much does the brick weigh?",
        "intuitive_wrong": "1.5",
        "correct": "2",
        "answer_unit": "kg",
        "explanation": "Let brick = b. b = 1 + b/2 → b/2 = 1 → b = 2 kg. Intuitive answer 1.5 treats 'half a brick' as 0.5 kg.",
        "difficulty": "medium",
        "cognitive_trap": "treating 'half a brick' as half a kilogram instead of half the unknown weight",
    },
    {
        "id": "CRT19",
        "question": "Three friends split a restaurant bill of $30, each paying $10. The waiter realizes he overcharged by $5 and returns it. The friends each take $1 back and leave a $2 tip. Now each friend paid $9. 3 × $9 = $27, plus the $2 tip = $29. Where did the missing dollar go?",
        "intuitive_wrong": "lost",
        "correct": "0",
        "answer_unit": "dollars",
        "explanation": "No dollar is missing. The $27 INCLUDES the tip. $27 = $25 (bill) + $2 (tip). The $30 = $25 + $3 (returned) + $2 (tip). Adding $2 to $27 is the error — it double-counts.",
        "difficulty": "hard",
        "cognitive_trap": "the misleading arithmetic framing makes you search for a missing dollar that doesn't exist",
    },
    {
        "id": "CRT20",
        "question": "If you have a 3-gallon jug and a 5-gallon jug, and you need exactly 4 gallons, what is the MINIMUM number of pour or fill actions needed? (Fill from tap, empty, or pour between jugs each count as one action.)",
        "intuitive_wrong": "5",
        "correct": "6",
        "answer_unit": "actions",
        "explanation": "Fill 5, pour into 3 (leaving 2 in 5), empty 3, pour 2 into 3, fill 5, pour from 5 into 3 (1 gallon fills it, leaving 4 in 5). That's 6 actions.",
        "difficulty": "hard",
        "cognitive_trap": "underestimating the steps needed, or guessing a round number",
    },
]


In [ ]:
# Remove the data import line since data is inlined above
"""
Executive Functions Benchmark 5: Cognitive Reflection Test (CRT)

Tests the ability to override intuitive-but-wrong responses (System 1)
with deliberate reasoning (System 2). This measures response inhibition,
a core component of executive function.

Cognitive Science Basis:
- Frederick (2005): The Cognitive Reflection Test
- Kahneman (2011): System 1 (fast, intuitive) vs System 2 (slow, deliberate)
- Miyake et al. (2000): Inhibition as a core executive function
- Toplak et al. (2011): CRT correlates with rational thinking ability

Protocol:
1. Present 12 novel CRT-style questions (not from published tests)
2. Each has a compelling intuitive-but-wrong answer
3. Model provides numerical/short answer + confidence (0-100)
4. Score: correct answers that resist the intuitive trap

Metrics:
- Accuracy: proportion of correct (System 2) answers
- Intuitive trap rate: proportion of intuitive-wrong answers
- Deliberation score: accuracy weighted by difficulty
- Confidence calibration: are correct answers higher-confidence?

Score = 0.40 * accuracy + 0.30 * (1 - trap_rate) + 0.20 * difficulty_bonus + 0.10 * calibration

Shortcut Resistance:
- Novel items (not from Frederick 2005 or published CRTs)
- Each item has a SPECIFIC intuitive wrong answer — we check if the model
  falls for it vs. gets a different wrong answer vs. gets it right
- Difficulty stratification reveals genuine reasoning vs. memorization
"""

import kaggle_benchmarks as kbench
from dataclasses import dataclass
import numpy as np
import re
import json



# ─── Structured Output Schema ──────────────────────────────────────

@dataclass
class CRTResponse:
    """Model's answer to a CRT question."""
    answer: str        # The answer (number or short text)
    confidence: int    # 0-100 confidence
    reasoning: str     # Explanation of thought process


# ─── Answer Checking ────────────────────────────────────────────────

def normalize_answer(answer: str) -> str:
    """Normalize an answer for comparison."""
    answer = answer.strip().lower()
    # Remove common prefixes
    for prefix in ['$', '£', '€']:
        answer = answer.replace(prefix, '')
    # Remove trailing units
    answer = re.sub(r'\s*(dollars?|cents?|minutes?|days?|sheep|position|percent|%|leaves?|times?|name).*$', '', answer, flags=re.IGNORECASE)
    answer = answer.strip().rstrip('.')
    return answer


def check_answer(model_answer: str, correct: str, intuitive_wrong: str):
    """
    Check if answer is correct, intuitively wrong, or other wrong.
    Returns: 'correct', 'intuitive_trap', or 'other_wrong'
    """
    norm_model = normalize_answer(str(model_answer))
    norm_correct = normalize_answer(str(correct))
    norm_intuitive = normalize_answer(str(intuitive_wrong))

    # Check for correct
    if norm_model == norm_correct:
        return 'correct'
    # Try numeric comparison
    try:
        if abs(float(norm_model) - float(norm_correct)) < 0.01:
            return 'correct'
    except (ValueError, TypeError):
        pass
    # Check for special cases
    if norm_correct == 'emily' and 'emily' in norm_model:
        return 'correct'

    # Check for intuitive trap
    if norm_model == norm_intuitive:
        return 'intuitive_trap'
    try:
        if abs(float(norm_model) - float(norm_intuitive)) < 0.01:
            return 'intuitive_trap'
    except (ValueError, TypeError):
        pass

    return 'other_wrong'


# ─── The Benchmark Task ────────────────────────────────────────────

@kbench.task(name="exec_func_crt")
def exec_func_crt(llm) -> float:
    """
    Cognitive Reflection Test Benchmark.

    Tests inhibition of intuitive-but-wrong answers in favor of
    deliberate reasoning. A core executive function measure.

    Score = 0.40 * accuracy + 0.30 * (1 - trap_rate) + 0.20 * difficulty_bonus + 0.10 * calibration

    Cognitive Science: Frederick (2005), Kahneman (2011).
    Human accuracy: ~30% (general public), ~50% (MIT students).
    """
    results = []
    difficulty_correct = {"easy": [], "medium": [], "hard": []}

    for item in CRT_ITEMS:
        prompt = (
            f"Please answer this question. Give ONLY the numerical answer "
            f"(or a short phrase if non-numerical), your confidence level (0-100), "
            f"and a brief explanation of your reasoning.\n\n"
            f"Question: {item['question']}\n\n"
            f"Think carefully before answering."
        )

        with kbench.chats.new(f"crt_{item['id']}"):
            try:
                response = llm(prompt, response_format=CRTResponse)
                answer = response.answer
                confidence = max(0, min(100, response.confidence))
                reasoning = response.reasoning
            except Exception:
                raw = llm(prompt)
                answer = raw.strip()
                confidence = 50
                reasoning = ""

        verdict = check_answer(answer, item['correct'], item['intuitive_wrong'])
        is_correct = verdict == 'correct'
        is_trap = verdict == 'intuitive_trap'

        result = {
            "id": item["id"],
            "difficulty": item["difficulty"],
            "model_answer": str(answer)[:100],
            "correct_answer": item["correct"],
            "intuitive_wrong": item["intuitive_wrong"],
            "verdict": verdict,
            "confidence": confidence,
            "cognitive_trap": item["cognitive_trap"],
        }
        results.append(result)
        difficulty_correct[item["difficulty"]].append(1.0 if is_correct else 0.0)

    # ── Compute Metrics ──

    n_correct = sum(1 for r in results if r["verdict"] == "correct")
    n_trap = sum(1 for r in results if r["verdict"] == "intuitive_trap")
    n_other = sum(1 for r in results if r["verdict"] == "other_wrong")

    accuracy = n_correct / len(results)
    trap_rate = n_trap / len(results)

    # Difficulty bonus: harder items worth more
    diff_weights = {"easy": 1.0, "medium": 1.5, "hard": 2.0}
    weighted_correct = 0
    weighted_total = 0
    for diff, scores in difficulty_correct.items():
        w = diff_weights[diff]
        weighted_correct += sum(s * w for s in scores)
        weighted_total += len(scores) * w
    difficulty_bonus = weighted_correct / weighted_total if weighted_total > 0 else 0

    # Calibration: are correct answers higher-confidence than wrong?
    correct_confs = [r["confidence"] for r in results if r["verdict"] == "correct"]
    wrong_confs = [r["confidence"] for r in results if r["verdict"] != "correct"]
    if correct_confs and wrong_confs:
        calibration = min(1.0, max(0.0,
            (np.mean(correct_confs) - np.mean(wrong_confs)) / 100 + 0.5
        ))
    else:
        calibration = 0.5  # No signal

    # ── Composite Score ──
    score = (
        0.40 * accuracy +
        0.30 * (1 - trap_rate) +
        0.20 * difficulty_bonus +
        0.10 * calibration
    )
    score = round(float(np.clip(score, 0, 1)), 4)

    # ── Log ──
    kbench.log({
        "benchmark": "Cognitive Reflection Test",
        "n_items": len(results),
        "accuracy": round(accuracy, 4),
        "intuitive_trap_rate": round(trap_rate, 4),
        "other_wrong_rate": round(n_other / len(results), 4),
        "difficulty_bonus": round(difficulty_bonus, 4),
        "calibration": round(calibration, 4),
        "composite_score": score,
        "difficulty_breakdown": {
            d: round(np.mean(s), 4) if s else 0
            for d, s in difficulty_correct.items()
        },
        "per_item": results,
    })

    # ── Display ──
    print(f"\n{'='*60}")
    print(f"COGNITIVE REFLECTION TEST RESULTS")
    print(f"{'='*60}")
    for r in results:
        icon = "✓" if r["verdict"] == "correct" else ("⚠" if r["verdict"] == "intuitive_trap" else "✗")
        print(f"  {icon} {r['id']} [{r['difficulty']:6s}] [{r['confidence']:3d}%] "
              f"Got: {r['model_answer'][:20]:20s} Correct: {r['correct_answer']:10s} "
              f"Trap: {r['intuitive_wrong']:5s} → {r['verdict']}")

    print(f"\n--- Summary ---")
    print(f"Accuracy:           {accuracy:.2%} ({n_correct}/{len(results)})")
    print(f"Intuitive trap rate: {trap_rate:.2%} ({n_trap}/{len(results)})")
    print(f"Other wrong:         {n_other/len(results):.2%}")
    print(f"Difficulty bonus:    {difficulty_bonus:.4f}")
    print(f"Calibration:         {calibration:.4f}")
    print(f"Composite score:     {score:.4f}")

    return score


exec_func_crt.run(llm=kbench.llm)
